In [ ]:
from __future__ import annotations

import argparse
import shutil
from pathlib import Path

import numpy as np
import s2fft


def resample_mw_to_mwss(map_path: Path, output_path: Path) -> None:
    m = np.load(map_path)
    m = m.reshape((1500, 2999, 5)).astype(np.float32)

    m_mwss = np.stack(
        [
            np.array(s2fft.utils.resampling.mw_to_mwss(m[..., c], L=1500)).real
            for c in range(5)
        ],
        axis=-1,
    )

    np.save(output_path, m_mwss.astype(np.float32, copy=False))




In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(
        description="Bulk-resample spherical maps from MW to MWSS."
    )
    parser.add_argument(
        "--source-dir",
        type=Path,
        default=Path(
            "/projects/u6pf/brianycc/spherical_maps/"
            "map_compression_samples_5_tomo_bins_flm_1500_batch_1"
        ),
        help="Directory containing map_*.npy and params.npy",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path(
            "/projects/u6pf/brianycc/spherical_maps/"
            "map_compression_samples_5_tomo_bins_flm_1500_batch_1_mwss"
        ),
        help="Directory where MWSS-resampled files will be written",
    )
    args = parser.parse_args()

    source_dir: Path = args.source_dir
    output_dir: Path = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    params_src = source_dir / "params.npy"
    params_dst = output_dir / "params.npy"
    if params_src.exists() and not params_dst.exists():
        shutil.copy2(params_src, params_dst)

    map_files = sorted(source_dir.glob("map_*.npy"))
    if not map_files:
        raise FileNotFoundError(f"No map_*.npy files found in {source_dir}")

    skipped = 0
    converted = 0

    for map_path in map_files:
        output_path = output_dir / map_path.name
        if output_path.exists():
            skipped += 1
            continue
        resample_mw_to_mwss(map_path, output_path)
        converted += 1

    print(
        f"Converted {converted} maps, skipped {skipped} existing maps "
        f"from {source_dir} to {output_dir}"
    )


if __name__ == "__main__":
    main()